In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import glob
import matplotlib.pyplot as plt

In [2]:
files = glob.glob(
    "../Temperature/*.csv"
)
print(files)

['../Temperature\\Helsinki-Vantaa airport_ 1.1.2023 - 30.6.2023.csv', '../Temperature\\Helsinki-Vantaa airport_ 1.1.2024 - 30.6.2024.csv', '../Temperature\\Helsinki-Vantaa airport_ 1.1.2025 - 30.6.2025.csv', '../Temperature\\Helsinki-Vantaa airport_ 1.11.2024 - 31.12.2024.csv', '../Temperature\\Helsinki-Vantaa airport_ 1.7.2023 - 31.12.2023.csv', '../Temperature\\Helsinki-Vantaa airport_ 1.7.2024 - 31.10.2024.csv', '../Temperature\\Helsinki-Vantaa airport_ 1.7.2025 - 31.12.2025.csv']


In [3]:
df_list=[]

for file in files:

    temp=pd.read_csv(file)

    df_list.append(temp)

temp_df=pd.concat(df_list)

print(temp_df.shape)

(157806, 6)


In [4]:
temp_df.info();temp_df.describe();temp_df.head(20);temp_df.sample(20)

<class 'pandas.DataFrame'>
Index: 157806 entries, 0 to 26495
Data columns (total 6 columns):
 #   Column                     Non-Null Count   Dtype 
---  ------                     --------------   ----- 
 0   Observation station        157806 non-null  str   
 1   Year                       157806 non-null  int64 
 2   Month                      157806 non-null  int64 
 3   Day                        157806 non-null  int64 
 4   Time [Local time]          157806 non-null  str   
 5   Air temperature mean [°C]  157806 non-null  object
dtypes: int64(3), object(1), str(2)
memory usage: 8.4+ MB


,Observation station,Year,Month,Day,Time [Local time],Air temperature mean [°C]
8413,Vantaa Helsinki-Vantaa airport,2024,8,28,10:10,19.3
9675,Vantaa Helsinki-Vantaa airport,2023,3,9,04:30,-11.5
1620,Vantaa Helsinki-Vantaa airport,2023,1,12,06:00,0.6
24226,Vantaa Helsinki-Vantaa airport,2023,12,16,05:40,-4.4
26148,Vantaa Helsinki-Vantaa airport,2024,6,30,15:00,21.1
16732,Vantaa Helsinki-Vantaa airport,2024,4,26,05:40,0.8
18856,Vantaa Helsinki-Vantaa airport,2023,11,8,22:40,-1.6
336,Vantaa Helsinki-Vantaa airport,2024,7,3,08:00,14.6
12543,Vantaa Helsinki-Vantaa airport,2023,3,29,03:30,-4.9
11208,Vantaa Helsinki-Vantaa airport,2024,9,16,20:00,14.7


In [ ]:
#check every cell in the dataframe to see if there are any missing values
temp_df.isnull().sum()

Observation station          0
Year                         0
Month                        0
Day                          0
Time [Local time]            0
Air temperature mean [°C]    0
dtype: int64

In [ ]:
#check the data type of the 'Air temperature mean [°C]' column
temp_df["Air temperature mean [°C]"].dtype

dtype('O')

In [ ]:
# convert the 'Air temperature mean [°C]' column to numeric, coercing errors to NaN, and then forward fill the NaN values
temp_df['Air temperature mean [°C]'] = pd.to_numeric(temp_df['Air temperature mean [°C]'], errors='coerce')

# forward fill the NaN values in the 'Air temperature mean [°C]' column
temp_df['Air temperature mean [°C]'] = temp_df['Air temperature mean [°C]'].ffill()

In [8]:
temp_df.info()

<class 'pandas.DataFrame'>
Index: 157806 entries, 0 to 26495
Data columns (total 6 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Observation station        157806 non-null  str    
 1   Year                       157806 non-null  int64  
 2   Month                      157806 non-null  int64  
 3   Day                        157806 non-null  int64  
 4   Time [Local time]          157806 non-null  str    
 5   Air temperature mean [°C]  157806 non-null  float64
dtypes: float64(1), int64(3), str(2)
memory usage: 8.4 MB


In [ ]:
# create a new column 'datetime_str' by concatenating the 'Year', 'Month', 'Day', and 'Time [Local time]' columns, and then convert this new column to a datetime object in a new column called 'datetime'
temp_df['datetime_str'] = temp_df['Year'].astype(str) + '-' + temp_df['Month'].astype(str) + '-' + temp_df['Day'].astype(str) + ' ' + temp_df['Time [Local time]']

# convert the 'datetime_str' column to a datetime object in a new column called 'datetime'
temp_df['datetime'] = pd.to_datetime(temp_df['datetime_str'])

In [ ]:
# change the index to datetime and reset the index to get a clean hourly temperature dataframe
temp_df.set_index('datetime', inplace=True)

# set 10 mins interval to get the hourly average temperature
hourly_temp = temp_df['Air temperature mean [°C]'].resample('h').mean().reset_index()

In [ ]:
temp_df.info();temp_df.describe();

<class 'pandas.DataFrame'>
DatetimeIndex: 157806 entries, 2023-01-01 00:00:00 to 2025-12-31 23:50:00
Data columns (total 7 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Observation station        157806 non-null  str    
 1   Year                       157806 non-null  int64  
 2   Month                      157806 non-null  int64  
 3   Day                        157806 non-null  int64  
 4   Time [Local time]          157806 non-null  str    
 5   Air temperature mean [°C]  157806 non-null  float64
 6   datetime_str               157806 non-null  str    
dtypes: float64(1), int64(3), str(3)
memory usage: 9.6 MB


,Year,Month,Day,Air temperature mean [°C]
count,157806.000000,157806.000000,157806.000000,157806.000000
mean,2024.000000,6.522300,15.731151,6.943735
std,0.816127,3.448989,8.800729,9.316474
min,2023.000000,1.000000,1.000000,-26.500000
25%,2023.000000,4.000000,8.000000,0.200000
50%,2024.000000,7.000000,16.000000,6.100000
75%,2025.000000,10.000000,23.000000,14.700000
max,2025.000000,12.000000,31.000000,29.700000


In [15]:
temp_df.sample(5)

,Observation station,Year,Month,Day,Time [Local time],Air temperature mean [°C],datetime_str
datetime,,,,,,,
2023-05-21 21:50:00,Vantaa Helsinki-Vantaa airport,2023,5,21,21:50,17.1,2023-5-21 21:50
2025-10-23 20:40:00,Vantaa Helsinki-Vantaa airport,2025,10,23,20:40,6.8,2025-10-23 20:40
2025-06-06 04:20:00,Vantaa Helsinki-Vantaa airport,2025,6,6,04:20,7.9,2025-6-6 04:20
2024-11-03 07:20:00,Vantaa Helsinki-Vantaa airport,2024,11,3,07:20,0.9,2024-11-3 07:20
2025-06-01 17:00:00,Vantaa Helsinki-Vantaa airport,2025,6,1,17:00,19.5,2025-6-1 17:00


In [17]:
# set 10 mins interval to get the hourly average temperature
hourly_temp = temp_df['Air temperature mean [°C]'].resample('h').mean()

# change the index to datetime and reset the index to get a clean hourly temperature dataframe
hourly_temp_df = hourly_temp.reset_index()

# save the hourly temperature dataframe to a csv file
hourly_temp_df.to_csv("../Temperature/Temperature_hourly_clean.csv", index=False)